# DPR Retriever 학습

**MINI 모드** (기본값): NQ 5K, 배치 32, 5 epochs → 동작 확인용  
**풀 학습**: `MINI = False` 로 변경 → NQ 전체 58K, 배치 128, 40 epochs

## 0. 환경 설정

In [ ]:
import sys, os

!git clone https://github.com/Teddykwj/dpr-reproduction.git 2>/dev/null || \
 git -C /kaggle/working/dpr-reproduction pull

for name in ['dpr-reproduction', 'DPR-REPRODUCTION']:
    candidate = f'/kaggle/working/{name}'
    if os.path.isdir(candidate):
        REPO_ROOT = candidate
        break

sys.path.insert(0, REPO_ROOT)
print('REPO_ROOT:', REPO_ROOT)

## 1. 설정

In [ ]:
import torch

# ── 여기만 바꾸면 됨 ─────────────────────────────────
MINI = True   # False 로 바꾸면 풀 학습
# ────────────────────────────────────────────────────

BATCH_SIZE  = 32    if MINI else 128
NUM_EPOCHS  = 5     if MINI else 40
MAX_SAMPLES = 5_000 if MINI else None

# 데이터 경로 (Kaggle Dataset 마운트)
NQ_TRAIN = '/kaggle/input/datasets/teddykwj/dpr-reproduction-data/data/nq/biencoder-nq-train.json'
NQ_DEV   = '/kaggle/input/datasets/teddykwj/dpr-reproduction-data/data/nq/biencoder-nq-dev.json'
CKPT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {DEVICE}, mini: {MINI}, batch: {BATCH_SIZE}, epochs: {NUM_EPOCHS}')

## 2. 데이터 로드

In [ ]:
from torch.utils.data import DataLoader, Subset
from transformers import BertTokenizerFast
from src.data.dataset import NQDataset
from src.data.collator import DPRCollator

tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
collator  = DPRCollator(tokenizer)

train_dataset = NQDataset(NQ_TRAIN, tokenizer)
dev_dataset   = NQDataset(NQ_DEV,   tokenizer)

if MAX_SAMPLES:
    train_dataset.data = train_dataset.data[:MAX_SAMPLES]

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collator, num_workers=2, pin_memory=True)
dev_loader   = DataLoader(dev_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collator, num_workers=2, pin_memory=True)

print(f'train: {len(train_dataset):,}  dev: {len(dev_dataset):,}')
print(f'train steps/epoch: {len(train_loader)}')

## 3. 모델 / 옵티마이저 / 스케줄러

In [ ]:
from transformers import get_linear_schedule_with_warmup
from src.models.biencoder import BiEncoder

model     = BiEncoder().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

total_steps   = len(train_loader) * NUM_EPOCHS
warmup_steps  = int(total_steps * 0.1)
scheduler     = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

print(f'total steps: {total_steps:,}  warmup: {warmup_steps:,}')

## 4. 학습 루프

In [ ]:
from tqdm import tqdm
from src.models.loss import in_batch_negative_loss


def run_epoch(loader, training: bool):
    model.train() if training else model.eval()
    total_loss, steps = 0.0, 0

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch in tqdm(loader, leave=False):
            q_emb, p_emb = model(
                batch['q_input_ids'].to(DEVICE),
                batch['q_attention_mask'].to(DEVICE),
                batch['q_token_type_ids'].to(DEVICE),
                batch['p_input_ids'].to(DEVICE),
                batch['p_attention_mask'].to(DEVICE),
                batch['p_token_type_ids'].to(DEVICE),
            )
            h_emb = model.passage_encoder(
                batch['h_input_ids'].to(DEVICE),
                batch['h_attention_mask'].to(DEVICE),
                batch['h_token_type_ids'].to(DEVICE),
            )

            loss = in_batch_negative_loss(q_emb, p_emb, h_emb)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                scheduler.step()

            total_loss += loss.item()
            steps += 1

    return total_loss / steps


best_dev_loss = float('inf')

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = run_epoch(train_loader, training=True)
    dev_loss   = run_epoch(dev_loader,   training=False)

    print(f'epoch {epoch:02d}  train_loss: {train_loss:.4f}  dev_loss: {dev_loss:.4f}')

    # best dev loss 기준으로 체크포인트 저장
    if dev_loss < best_dev_loss:
        best_dev_loss = dev_loss
        torch.save(model.state_dict(), f'{CKPT_DIR}/best.pt')
        print(f'  → checkpoint saved (dev_loss: {dev_loss:.4f})')

print('학습 완료')